In [73]:
import numpy as np
import random

# Utilization of hospital beds during epidemics

## Primary task

In [74]:
### 
# Is slow for some reason
####

"""
def arr_per_day(t, lambda_function, id_process): 
    time = 0
    arrivals = [(id_process,(t-1))]
    while True: 
        wait = np.random.exponential(lambda_function(t))
        time += wait 

        if time > 1:
            break
        
        arrivals.append((id_process,time + (t-1)))
    
    return arrivals

def arr_per_year(days):
    all_events = []
    for t in range(1,days+1):
        process_1 = arr_per_day(t, lambda_1, 1)
        process_2 = arr_per_day(t, lambda_2, 2)
        process_3 = arr_per_day(t, lambda_3, 3)

        # Merge daily events 

        all_events += process_1 + process_2 + process_3
    
    # sort by time (assumes time is index 2)
    all_events = sorted(all_events, key=lambda x: x[1])

    return all_events

"""


'\ndef arr_per_day(t, lambda_function, id_process): \n    time = 0\n    arrivals = [(id_process,(t-1))]\n    while True: \n        wait = np.random.exponential(lambda_function(t))\n        time += wait \n\n        if time > 1:\n            break\n        \n        arrivals.append((id_process,time + (t-1)))\n    \n    return arrivals\n\ndef arr_per_year(days):\n    all_events = []\n    for t in range(1,days+1):\n        process_1 = arr_per_day(t, lambda_1, 1)\n        process_2 = arr_per_day(t, lambda_2, 2)\n        process_3 = arr_per_day(t, lambda_3, 3)\n\n        # Merge daily events \n\n        all_events += process_1 + process_2 + process_3\n    \n    # sort by time (assumes time is index 2)\n    all_events = sorted(all_events, key=lambda x: x[1])\n\n    return all_events\n\n'

In [75]:
def arrivals_in_day(rate, t, Idx_for_process): 
    # Input: rate for arrival time, t is the day in the year, Idx_for_process is the type of patient
    # Output: list of a tuples with patient type in first entry and arrivaltime in the second entry. 


    # Initialize start of day and patients.
    time = 0
    patients = []

    # Let 0 patients arrive if rate is 0
    if rate <=0: 
        return []

    
    while True:
        time += np.random.exponential(1 / rate)

        # Check we are still within one day
        if time > 1:
            break

        # Append patient type and time for arrival
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    # Input: lami is the arrival rate function for ward i. 
    # Output: A list of tuples where the first entry in the tuple is the patient type and the last entry is the arrival time.
    
    # Initialize
    t =0
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []

    #Iterate over the days
    while t < 365: 
        # Find rates
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        # Simulate arrivals for all three patient types. 
        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1

    # Merge list to create one list of all arrivals in a year
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients

def lambda_1(t): 
    return -(1/3650)*t**2 + (1/10)*t 

def lambda_2(t):
    return 1/5*lambda_1(t)

def lambda_3(t):
    return 6

In [76]:
Patients = arrivals_year(lambda_1, lambda_2,lambda_3)

In [77]:
Patients[-1]

(3, 364.59127239768856)

In [ ]:
# System of wards - Do not understand how is beds made in in B an C, how many to begynd 
def system(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0
   

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))

        LOS_A = []
        LOS_B = []
        LOS_C = []

        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)
    

            else:
                # Reallocate patient
                blocked_A += 1
        
        
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)
                

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    LOS_B.append(LOS)
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))

            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


In [79]:
np.random.seed(42)
bedsA = 45 
bedsB = 15 #shoulden't this be 0? 
bedsC = 15
relocated_A, blocked_B, relocated_C, mean_occupied_A, mean_occupied_B, mean_occupied_C  = system(bedsA,bedsB,bedsC, Patients)

relocated_A, blocked_B, relocated_C, mean_occupied_A, mean_occupied_B, mean_occupied_C 

(541,
 103,
 1560,
 np.float64(0.8333633821048009),
 np.float64(0.7515983634976771),
 np.float64(0.9734137715831079))

## Primary performance measures

In [ ]:
# Crude monte carlo estimator
def probs(bedsA,bedsB,bedsC,n): #still do not get the input 
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lambda_1, lambda_2,lambda_3)
        A, B, C, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(A/type_A)
        frac_B.append(B/type_B)
        frac_C.append(C/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)


    
    all = A+B+C
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

    

In [81]:
np.random.seed(42)
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(15,15,45,1000)

In [82]:
# Probability of blocked upon arrival 
fracA,fracB,fracC

(np.float64(0.7196318266389414),
 np.float64(0.2119614977497156),
 np.float64(0.213229148138691))

In [83]:
# Blocked patiens (relocated)
meanA, meanB,meanC, meanall


(np.float64(1589.0), np.float64(87.0), np.float64(400.0), np.float64(2076.0))

In [84]:
# mean fraction of beds that are utilized each ward
meanbedA,meanbedB,meanbedC

(np.float64(0.9375306715764559),
 np.float64(0.7461978290110446),
 np.float64(0.9305957450613709))

## Sensitivity analysis

In [151]:
# We now need to optimize the bed distribution based on minimizing the sum of (we use all) relocated patients.
# Monte carlo estimator 
def sum_relocated_MC(bedsA,bedsB,bedsC,n):
    A = []
    B = []
    C = []

    for _ in range(n):
        X = arrivals_year(lambda_1,lambda_2,lambda_3)
        A_blok, B_blok, C_blok, _, _, _ = system(bedsA,bedsB,bedsC, X)
        A.append(A_blok)
        B.append(B_blok)
        C.append(C_blok)
    all = np.array(A) + np.array(B) + np.array(C)

    return np.mean(all)

In [152]:
sum_relocated_MC(15,15,45,100)

np.float64(2162.9)

In [153]:
# System of wards - Do not understand how is beds made in in B an C, how many to begynd 
def system_CV(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0


    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []
    LOS_A = []
    LOS_B = []
    LOS_C = []
    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))

    
       
        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)
    

            else:
                # Reallocate patient
                blocked_A += 1
        
        
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)
                

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    LOS_B.append(LOS)
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))

            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)

            else:
                # Reallocate patient
                blocked_C += 1

    return blocked_A,blocked_B,blocked_C, np.mean(LOS_A), np.mean(LOS_B), np.mean(LOS_C)

In [170]:
patients = []
for i in range(100):
    patients.append(arrivals_year(lambda_1,lambda_2,lambda_3))

In [176]:
# We now need to optimize the bed distribution based on minimizing the sum of (we use all) relocated patients.
# Control variate 
LOS_mu_A = 8 
LOS_mu_B = 12
LOS_mu_C = 10 

def sum_relocated_CV(bedsA,bedsB,bedsC,n, patients):
    A = []
    B = []
    C = []

    LOS_A = []
    LOS_B = []
    LOS_C = []

    for i in range(n):
        X = patients[i]
        a, b, c, LOS_a, LOS_b, LOS_c = system_CV(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)

        LOS_A.append(LOS_a)
        LOS_B.append(LOS_b)
        LOS_C.append(LOS_c)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)

    LOS_A = np.array(LOS_A)
    LOS_B = np.array(LOS_B)
    LOS_C = np.array(LOS_C)
        
    c_A = np.cov(A, LOS_A, ddof=1)[0,1] / np.var(LOS_A, ddof=1)
    Y_A = A + c_A*(LOS_A - LOS_mu_A)

    c_B = np.cov(B, LOS_B, ddof=1)[0,1] / np.var(LOS_B, ddof=1)
    Y_B = B + c_B*(LOS_B - LOS_mu_B)

    c_C = np.cov(C, LOS_C, ddof=1)[0,1] / np.var(LOS_C, ddof=1)
    Y_C = C + c_C*(LOS_C - LOS_mu_C)

    return np.mean(Y_A) + np.mean(Y_B) + np.mean(Y_C)

In [177]:
sum_relocated_CV(17,14,44,100, patients)

np.float64(2014.185209945619)

In [163]:
# Different bed distributions 
bed_scenarios = [
    [33, 7, 35],  # 1. Traffic-Proportional Base
    [25, 25, 25], # 2. Equal Split
    [10, 10, 55], # 3. Current Setup (Ward C Favored)
    [50, 5, 20],  # 4. Aggressive Ward A Focus
    [65, 5, 5],    # 5. Minimum-Bed Stress Test
    [45, 0, 30],   # 6. Delete ward B
]
n = 100
for bedA, bedB, bedC in bed_scenarios:
    mean_block = sum_relocated_MC(bedA,bedB,bedC,n)
    print(f"Monte Carlo estimator, n = {n}")
    print(f"Mean of blocked patientens(relocated): {mean_block}")
    print(f"Bed distribution A: {bedA}, B: {bedB}, C: {bedC}")
    print()

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2123.98
Bed distribution A: 33, B: 7, C: 35

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2334.39
Bed distribution A: 25, B: 25, C: 25

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2228.63
Bed distribution A: 10, B: 10, C: 55

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2300.56
Bed distribution A: 50, B: 5, C: 20

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2582.47
Bed distribution A: 65, B: 5, C: 5



/var/folders/x2/5q06y6lx7pl1m_kc24264lbh0000gn/T/ipykernel_81370/1609428975.py:87: RuntimeWarning: invalid value encountered in scalar divide
  return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2278.54
Bed distribution A: 45, B: 0, C: 30



In [179]:
# Different bed distributions 
bed_scenarios = [
    [33, 7, 35],  # 1. Traffic-Proportional Base
    [25, 25, 25], # 2. Equal Split
    [10, 10, 55], # 3. Current Setup (Ward C Favored)
    [50, 5, 20],  # 4. Aggressive Ward A Focus
    [65, 5, 5],    # 5. Minimum-Bed Stress Test
    [45, 0, 30],   # 6. Delete ward B
]
n = 100
for bedA, bedB, bedC in bed_scenarios:
    mean_block = sum_relocated_CV(bedA,bedB,bedC,n, patients)
    print(f"Control variate estimator, n = {n}")
    print(f"Mean of blocked patientens(relocated): {mean_block}")
    print(f"Bed distribution A: {bedA}, B: {bedB}, C: {bedC}")
    print()

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2021.5338574143302
Bed distribution A: 33, B: 7, C: 35

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2144.744474175355
Bed distribution A: 25, B: 25, C: 25

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2223.1315572139692
Bed distribution A: 10, B: 10, C: 55

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2177.7366278282516
Bed distribution A: 50, B: 5, C: 20

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2476.5042009052377
Bed distribution A: 65, B: 5, C: 5

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2294.189834564374
Bed distribution A: 45, B: 0, C: 30



### Task 1: Find optimal bed distribution
Based on sum of relocated patients 

In [168]:
# Minimize the sum of relocated patients for different bed distribution

best_bed= []

for run in range(10):

    best_val = float("inf")
    best_alloc = None

    for _ in range(100):
        A = random.randint(1, 73)
        B = random.randint(1, 75 - A)
        C = 75 - A - B

        if C <=0: 
            continue

        val = sum_relocated_CV(A, B, C,10)

        if val < best_val:
            best_val = val
            best_alloc = (A, B, C)
        
    print(run)
    best_bed.append((best_alloc, best_val))

0
1
2
3
4
5
6
7
8
9


In [169]:
best_bed

[((29, 20, 26), np.float64(1572.9611711217422)),
 ((26, 19, 30), np.float64(1794.4511888138004)),
 ((31, 10, 34), np.float64(1690.4057535954475)),
 ((24, 15, 36), np.float64(1821.3126292737868)),
 ((37, 13, 25), np.float64(1753.348902289788)),
 ((20, 19, 36), np.float64(1759.973959769432)),
 ((41, 11, 23), np.float64(1885.5635137506258)),
 ((23, 19, 33), np.float64(1850.2814813499583)),
 ((35, 10, 30), np.float64(1782.8912945741372)),
 ((33, 16, 26), np.float64(1689.5690122621145))]

In [ ]:
patients_100 = []
for i in range(100):
    patients_100.append(arrivals_year(lambda_1,lambda_2,lambda_3))


In [ ]:
def optimize_beds_stepwise(total_beds, patient_data, start_dist=[30, 10, 35], n):
    current_dist = list(start_dist)
    best_dist = list(start_dist)
    
    # Get baseline metrics using your updated function
    res = sum_relocated_CV(bedsA,bedsB,bedsC,n,patient_data)
    res = ward_flow_year(current_dist, patient_data)
    best_score = res[0] + res[1] + res[2] # Sum of relocated_A + blocked_B + relocated_C
    print(f"Starting Baseline {current_dist}: {best_score} total issues")

    improved = True
    while improved:
        improved = False
        neighbors = []
        for i in range(3):
            for j in range(3):
                # Ensure we don't drop a ward's bed count below 0
                if i != j and current_dist[i] > 0: 
                    test_dist = list(current_dist)
                    test_dist[i] -= 1
                    test_dist[j] += 1
                    neighbors.append(test_dist)
        
        for neighbor in neighbors:
            print(f"Testing adjustment: {neighbor}...")
            res_n = ward_flow_year(neighbor, patient_data)
            score = res_n[0] + res_n[1] + res_n[2]
            
            if score < best_score:
                best_score = score
                best_dist = neighbor
                improved = True
        
        if improved:
            current_dist = list(best_dist)
            print(f"Found better distribution: {current_dist} with {best_score} issues")
            
    return best_dist, best_score

# Run the stepwise optimization using your updated function
opt_dist, opt_score = optimize_beds_stepwise(75, flat_patients, start_dist=[45, 0, 30])
print(f"\nFinal Optimal Distribution: {opt_dist} with {opt_score} total problems")

In [ ]:
# Exponential length of stay distributial 

def system_exponential(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0
    

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))


        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.exponential(scale=8)
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_A += 1
            
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.exponential(scale=12)
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.exponential(scale=10)
            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC

# Crude monte carlo estimator
def probs_expontial(bedsA,bedsB,bedsC,n): #still do not get the input 
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for i in range(n):
        X = arrivals_year(lambda_1, lambda_2,lambda_3)
        A, B, C, bed_frac_A,bed_frac_B,bed_frac_C = system_exponential(bedsA,bedsB,bedsC, X)
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(A/type_A)
        frac_B.append(B/type_B)
        frac_C.append(C/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)


    
    all = A+B+C
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)